# Atividade: CNNs para Classificação

Neste notebook, iremos preparar nosso próprio dataset e treinar um modelo de classificação de imagens.

## Preparando os dados

Os dados desta atividade serão baixados da internet. Utilizaremos para isso buscadores comuns. Em seguida, dividiremos em treinamento e validação.

In [4]:
import os
import shutil
import random
from icrawler.builtin import GoogleImageCrawler, BingImageCrawler

### Adquirindo as Imagens

Utilizaremos o iCrawler para baixar imagens em buscadores através de termos especificados. Defina sua lista de classes.

In [5]:
def download_images_v1(keyword, folder, n_total=100):
    os.makedirs(folder, exist_ok=True)
    downloaded = len(os.listdir(folder))
    remaining = n_total - downloaded

    while downloaded < n_total:
        # crawler = GoogleImageCrawler(storage={'root_dir': folder})
        crawler = BingImageCrawler(storage={'root_dir': folder})
        crawler.crawl(keyword=keyword, max_num=remaining, file_idx_offset=downloaded)
        downloaded = len(os.listdir(folder))
        remaining = n_total - downloaded
        print(f"Downloaded {downloaded}/{n_total}")

    print("Download complete!")

A função original para download de imagens (renomeada para download_images_v1), não tratava adequadamente imagens repetidas nem execuções repetidas. Dessa forma reescrevemos a função de download para ignorar imagens repetidas e considerar o total de imagens desejado e o total de imagens já existentes no diretorio de destino. Dessa forma, a curadoria das imagens fica facilitada, permitindo a seleção de imagens de forma iterativa e mais dinâmica.
Além dessa função, criamos uma função auxiliar para identificar pares de imagens repetidas dentro de um diretório, permitindo limpar qualquer conjunto de imagens previamente carregadas.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path
from uuid import uuid4

from PIL import Image
import imagehash
from icrawler.builtin import BingImageCrawler
# from icrawler.builtin import GoogleImageCrawler


IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff"
}


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

def list_images(folder: Path):
    if not folder.exists():
        return []

    return [
        p for p in folder.iterdir()
        if is_image_file(p)
    ]


def count_images(folder: Path) -> int:
    return len(list_images(folder))


def calculate_phash(path: Path):
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            return imagehash.phash(img)
    except Exception:
        return None


def load_existing_hashes(folder: Path):
    hashes = []

    for path in list_images(folder):
        h = calculate_phash(path)
        if h is not None:
            hashes.append((path, h))

    return hashes


def is_duplicate(candidate_hash, existing_hashes, threshold=5):
    for _, existing_hash in existing_hashes:
        if candidate_hash - existing_hash <= threshold:
            return True

    return False


def next_image_name(folder: Path, label: str, extension: str) -> Path:
    while True:
        filename = f"{label}_{uuid4().hex[:12]}{extension.lower()}"
        target = folder / filename

        if not target.exists():
            return target


def download_images(
    keyword,
    folder,
    label=None,
    n_total=100,
    duplicate_threshold=5,
    batch_size=30,
    max_rounds=20
):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    if label is None:
        label = folder.name

    current_total = count_images(folder)

    if current_total >= n_total:
        print(
            f"Pasta já possui {current_total} imagens. "
            f"Nenhum download necessário para alvo n_total={n_total}."
        )
        return

    existing_hashes = load_existing_hashes(folder)

    print(f"Imagens já existentes: {current_total}")
    print(f"Meta final: {n_total}")
    print(f"Novas imagens necessárias: {n_total - current_total}")
    print(f"Hashes carregados: {len(existing_hashes)}")

    rounds = 0

    while current_total < n_total and rounds < max_rounds:
        rounds += 1

        remaining = n_total - current_total

        # Baixa mais do que o restante porque algumas serão descartadas.
        current_batch_size = max(batch_size, remaining * 3)

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            crawler = BingImageCrawler(storage={"root_dir": str(tmpdir)})
            # crawler = GoogleImageCrawler(storage={"root_dir": str(tmpdir)})

            crawler.crawl(
                keyword=keyword,
                max_num=current_batch_size
            )

            candidates = [
                p for p in tmpdir.iterdir()
                if is_image_file(p)
            ]

            accepted = 0
            discarded_duplicates = 0
            discarded_invalid = 0

            for candidate in candidates:
                if current_total >= n_total:
                    break

                candidate_hash = calculate_phash(candidate)

                if candidate_hash is None:
                    discarded_invalid += 1
                    continue

                if is_duplicate(
                    candidate_hash,
                    existing_hashes,
                    threshold=duplicate_threshold
                ):
                    discarded_duplicates += 1
                    continue

                extension = candidate.suffix.lower()

                if extension not in IMAGE_EXTENSIONS:
                    extension = ".jpg"

                target = next_image_name(folder, label, extension)

                shutil.move(str(candidate), str(target))

                existing_hashes.append((target, candidate_hash))
                current_total += 1
                accepted += 1

            print(
                f"Rodada {rounds}: "
                f"aceitas={accepted}, "
                f"duplicadas={discarded_duplicates}, "
                f"inválidas={discarded_invalid}, "
                f"total_final={current_total}/{n_total}"
            )

            if accepted == 0:
                print("Nenhuma imagem nova aceita nesta rodada. A busca pode estar saturada.")

    if current_total == n_total:
        print(f"Download completo. Pasta destino contém exatamente {n_total} imagens.")
    else:
        print(
            f"Encerrado com {current_total}/{n_total}. "
            f"Limite de rodadas atingido."
        )

In [15]:
search_terms = {
    # nome da classe: termo que será usado na busca
    "abelha": "single real bee on nature",
    "vespa": "single real wasp in nature"
}

for label, term in search_terms.items():
    download_images(term, f"data/insetos/{label}", n_total=100)

2026-05-10 18:38:17,109 - INFO - icrawler.crawler - start crawling...
2026-05-10 18:38:17,109 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-05-10 18:38:17,109 - INFO - feeder - thread feeder-001 exit
2026-05-10 18:38:17,110 - INFO - icrawler.crawler - starting 1 parser threads...
2026-05-10 18:38:17,111 - INFO - icrawler.crawler - starting 1 downloader threads...


Imagens já existentes: 99
Meta final: 100
Novas imagens necessárias: 1
Hashes carregados: 99


2026-05-10 18:38:17,911 - INFO - parser - parsing result page https://www.bing.com/images/async?q=single real bee on nature&first=0
2026-05-10 18:38:18,325 - INFO - downloader - image #1	https://cdn.pixabay.com/photo/2022/08/26/22/01/bee-7413333_1280.jpg
2026-05-10 18:38:18,572 - INFO - downloader - image #2	https://thumbs.dreamstime.com/b/bee-purple-flower-macro-park-nature-bee-purple-flower-macro-106388710.jpg
2026-05-10 18:38:18,690 - INFO - downloader - image #3	https://thumbs.dreamstime.com/z/single-bee-yellow-flower-macro-view-nectar-sucking-46059672.jpg
2026-05-10 18:38:18,838 - INFO - downloader - image #4	https://cdn.pixabay.com/photo/2020/06/05/17/41/bumble-bee-on-flower-5263906_1280.jpg
2026-05-10 18:38:19,807 - INFO - downloader - image #5	http://upload.wikimedia.org/wikipedia/commons/b/b5/Honey_bee_(Apis_mellifera).jpg
2026-05-10 18:38:20,031 - INFO - downloader - image #6	https://www.cbc.ca/natureofthings/content/legacy/BeesDiary_Close-up_1920.jpg
2026-05-10 18:38:20,088 

Rodada 1: aceitas=1, duplicadas=0, inválidas=0, total_final=100/100
Download completo. Pasta destino contém exatamente 100 imagens.
Pasta já possui 100 imagens. Nenhum download necessário para alvo n_total=100.


In [8]:
RANDOM_SEED = 42

### Treinamento e Validação

Dividiremos as imagens baixadas nas pastas `train` e `val`. Defina uma porcentagem.

In [ ]:
def split_train_val(root_dir, train_ratio=0.8, seed=RANDOM_SEED):
    random.seed(seed)

    train_dir = root_dir + "_split/train"
    val_dir = root_dir + "_split/val"

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    for class_name in os.listdir(root_dir):
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        images = [
            os.path.join(class_path, f)
            for f in os.listdir(class_path)
        ]

        images = [
            f for f in images
            if is_image_file(f)
        ]

        random.shuffle(images)

        n_train = int(len(images) * train_ratio)

        train_class_dir = os.path.join(train_dir, class_name)
        val_class_dir = os.path.join(val_dir, class_name)

        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(val_class_dir, exist_ok=True)

        for img in images[:n_train]:
            shutil.copy(
                img,
                os.path.join(train_class_dir, os.path.basename(img))
            )

        for img in images[n_train:]:
            shutil.copy(
                img,
                os.path.join(val_class_dir, os.path.basename(img))
            )

        print(f"{class_name}: {n_train} train, {len(images) - n_train} val")


split_train_val("data/insetos")

vespa: 80 train, 20 val
abelha: 80 train, 20 val


## Dataset

Implemente um Dataset PyTorch que carregue as imagens baixadas com suas respectivas classes. Aplique data augmentation e carregue em batches.

In [ ]:
import os
import random
from pathlib import Path

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF


class ImageClassificationDataset(Dataset):
    def __init__(
        self,
        root_dir,
        image_size=(224, 224),
        augment=False,
        augmentation_seed=None
    ):
        """
        root_dir:
            Diretório no formato:
                root_dir/
                  classe_1/
                    000001.jpg
                  classe_2/
                    000001.jpg

        image_size:
            Tupla no formato (altura, largura).

        augment:
            Se True, aplica augmentation determinística 8x.

        augmentation_seed:
            - None: apenas augmentation determinística.
            - int: augmentation determinística + augmentation aleatória reprodutível.
        """

        self.root_dir = Path(root_dir)
        self.image_size = image_size
        self.augment = augment
        self.augmentation_seed = augmentation_seed

        self.classes = sorted([
            p.name for p in self.root_dir.iterdir()
            if p.is_dir()
        ])

        self.class_to_idx = {
            class_name: idx
            for idx, class_name in enumerate(self.classes)
        }

        self.samples = self._load_samples()

        self.base_transform = transforms.Compose([
            transforms.Resize(self.image_size),
            transforms.ToTensor(),
        ])

        self.random_augmentation = transforms.Compose([
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
                saturation=0.10,
                hue=0.02
            ),
            transforms.RandomRotation(
                degrees=15
            ),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.05, 0.05),
                scale=(0.95, 1.05),
                shear=3
            ),
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 0.6)
            ),
        ])

        self.deterministic_factor = 8 if self.augment else 1

    def _is_image_file(self, path: Path) -> bool:
        return is_image_file(path)

    def _load_samples(self):
        samples = []

        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            class_idx = self.class_to_idx[class_name]

            image_files = sorted([
                p for p in class_dir.iterdir()
                if self._is_image_file(p)
            ], key=lambda p: p.name.lower())

            for image_path in image_files:
                samples.append((image_path, class_idx))

        return samples

    def __len__(self):
        return len(self.samples) * self.deterministic_factor

    def _apply_deterministic_augmentation(self, image, variant_idx):
        """
        variant_idx:
            0: original
            1: rot90
            2: rot180
            3: rot270
            4: espelho
            5: espelho + rot90
            6: espelho + rot180
            7: espelho + rot270
        """

        if variant_idx >= 4:
            image = TF.hflip(image)
            variant_idx -= 4

        if variant_idx == 1:
            image = TF.rotate(image, 90)
        elif variant_idx == 2:
            image = TF.rotate(image, 180)
        elif variant_idx == 3:
            image = TF.rotate(image, 270)

        return image

    def _apply_random_augmentation(self, image, index):
        if self.augmentation_seed is None:
            return image

        seed = self.augmentation_seed + index

        random.seed(seed)
        torch.manual_seed(seed)

        return self.random_augmentation(image)

    def __getitem__(self, index):
        if self.augment:
            sample_idx = index // self.deterministic_factor
            variant_idx = index % self.deterministic_factor
        else:
            sample_idx = index
            variant_idx = 0

        image_path, label = self.samples[sample_idx]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.augment:
            image = self._apply_deterministic_augmentation(image, variant_idx)
            image = self._apply_random_augmentation(image, index)

        image = self.base_transform(image)

        return image, label

### Criando datasets e loaders de treino e validação

In [18]:
train_dataset = ImageClassificationDataset(
    root_dir="data/insetos_split/train",
    image_size=(224, 224),
    augment=True,
    augmentation_seed=42
)

val_dataset = ImageClassificationDataset(
    root_dir="data/insetos_split/val",
    image_size=(224, 224),
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)



### Inspecionando imagens geradas

In [19]:
import shutil
from pathlib import Path

from torchvision.transforms.functional import to_pil_image


def export_dataset_for_inspection(
    dataset,
    output_dir="data/inspection",
    clean_output=True
):
    output_dir = Path(output_dir)

    if clean_output and output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    idx_to_class = {
        idx: class_name
        for class_name, idx in dataset.class_to_idx.items()
    }

    counters = {
        class_name: 0
        for class_name in dataset.classes
    }

    for i in range(len(dataset)):
        image_tensor, label = dataset[i]

        class_name = idx_to_class[label]
        class_dir = output_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        counters[class_name] += 1

        output_path = class_dir / f"{counters[class_name]:06d}.jpg"

        image = to_pil_image(image_tensor)
        image.save(output_path, quality=95)

    print("Exportação concluída.")

    for class_name in dataset.classes:
        print(f"{class_name}: {counters[class_name]} imagens")

In [20]:
inspection_dataset = ImageClassificationDataset(
    root_dir="data/insetos_split/train",
    image_size=(224, 224),
    augment=True,
    augmentation_seed=RANDOM_SEED
)

export_dataset_for_inspection(
    dataset=inspection_dataset,
    output_dir="data/inspection",
    clean_output=True
)

Exportação concluída.
abelha: 640 imagens
vespa: 640 imagens


## Definição do Modelo

Defina aqui o modelo que será utilizado, sendo implementação própria ou um modelo pré-treinado. Teste diversas arquiteturas diferentes e verifique qual delas tem melhor desempenho em validação.

In [7]:
# Seu código aqui

## Treinamento

Defina a função de custo e o otimizador do modelo. Em seguida, implemente o código de treinamento e treine-o. Ao final, exiba as curvas de treinamento e validação para a loss e a acurácia.

In [8]:
# Seu código aqui

## Inferência

Calcule algumas métricas como acurácia, matriz de confusão, etc. Em seguida, teste o modelo em novas imagens das classes correspondentes mas de outras fontes (outro buscador, fotos próprias, etc).

In [9]:
# Seu código aqui